# Assignment 10: Comprehensive Timed Challenge — Mock USAAIO Round 2 (100 points)

## INSTRUCTIONS

**This is a timed exam simulation. Set a timer for 4 hours (240 minutes).**

- This exam contains 3 sections, each worth ~33 points
- You may attempt the sections in any order
- Use the stated results from earlier parts to continue if you get stuck
- Write something for every part — partial credit is awarded
- You may use PyTorch documentation but no other resources

**Time management suggestion:** ~80 minutes per section, with 10 minutes for reading all problems first and 10 minutes for review at the end.

---

## Section A: Sparse Mixture-of-Experts Transformer (34 points)

### Background

In a standard Transformer FFN, every token is processed by the same feed-forward network. In a **Mixture-of-Experts (MoE)** layer, we have $E$ expert FFNs and a **router** that selects the top-$k$ experts for each token. This increases model capacity without proportionally increasing computation.

**Router:** Given token representation $h \in \mathbb{R}^d$, compute expert scores:
$$\text{scores} = \text{softmax}(hW_r) \in \mathbb{R}^E$$

Select the top-$k$ experts by score. The MoE output for this token is:
$$\text{MoE}(h) = \sum_{i \in \text{top-}k} g_i \cdot \text{Expert}_i(h)$$

where $g_i = \frac{\text{score}_i}{\sum_{j \in \text{top-}k} \text{score}_j}$ (renormalized scores).

**Load balancing loss:** To prevent all tokens from routing to the same expert, add an auxiliary loss:
$$\mathcal{L}_{\text{balance}} = E \cdot \sum_{i=1}^{E} f_i \cdot p_i$$

where $f_i = \frac{\text{number of tokens routed to expert } i}{\text{total tokens}}$ and $p_i = \frac{1}{T}\sum_{t=1}^{T} \text{score}_{t,i}$ (mean routing probability for expert $i$ across all tokens).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import math
import time

---

> **WARNING:** Do not modify any code outside of the designated solution areas.

---

### A.1: Router Implementation (6 points)

**[Coding]** Implement the router that selects the top-$k$ experts for each token.

- Input: $h \in \mathbb{R}^{B \times L \times d}$
- Compute scores: $\text{softmax}(h W_r)$ where $W_r \in \mathbb{R}^{d \times E}$
- Select top-$k$ experts per token
- Return: top-$k$ indices $(B, L, k)$, renormalized gates $(B, L, k)$, full scores $(B, L, E)$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class Router(nn.Module):
    def __init__(self, d_model, num_experts, top_k=2):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, h):
        """
        Returns:
            indices: (B, L, k) top-k expert indices
            gates: (B, L, k) renormalized gate values
            scores: (B, L, E) full softmax scores
        """
        pass  # YOUR CODE

""" END OF THIS PART """

### A.2: Expert FFN (4 points)

**[Coding]** Implement a single expert as a standard FFN: Linear($d$, $d_{ff}$) → GELU → Linear($d_{ff}$, $d$).

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class ExpertFFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x):
        pass  # YOUR CODE

""" END OF THIS PART """

### A.3: MoE Layer (8 points)

**[Coding]** Implement the Mixture-of-Experts layer.

For each token, run only the selected top-$k$ experts and combine their outputs with the gate values. Also compute the load balancing loss.

**Note:** A naive implementation loops over experts. For the exam, this is acceptable. The output shape is $(B, L, d)$.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class MoELayer(nn.Module):
    def __init__(self, d_model, d_ff, num_experts=8, top_k=2):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x):
        """
        Returns:
            output: (B, L, d)
            balance_loss: scalar
        """
        pass  # YOUR CODE

""" END OF THIS PART """

### A.4: MoE Transformer Block (6 points)

**[Coding]** Build a complete Transformer block that uses MoE instead of a standard FFN.

- Pre-norm LayerNorm → Multi-Head Self-Attention → Residual
- Pre-norm LayerNorm → MoE Layer → Residual
- Use `nn.MultiheadAttention` for the attention part
- Return both the output and the balance loss

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class MoETransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_experts=8, top_k=2):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x):
        pass  # YOUR CODE

""" END OF THIS PART """

### A.5: Shape and Parameter Analysis (4 points)

**[Non-coding]** For $d = 64$, $d_{ff} = 128$, $E = 8$, $k = 2$, $h = 4$:

1. (2 points) How many parameters does the MoE layer have compared to a standard FFN of the same dimensions? Express as a ratio.

2. (2 points) For a single token, how many FLOPs does the MoE layer use compared to the standard FFN? (Only $k$ experts are active.)

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

### A.6: Training (6 points)

**[Coding]** Train a MoE Transformer on a toy sequence classification task.

- 3-layer MoE Transformer with $d = 64$, $h = 4$, $d_{ff} = 128$, $E = 8$, $k = 2$
- Add embedding + mean pooling + classifier head
- Synthetic data: 800 training sequences, 200 test, length 16, vocab 100, 4 classes
- Total loss = classification loss + 0.01 * balance loss
- 20 epochs, Adam lr=1e-3, batch_size=32
- Report: final train/test accuracy and mean balance loss

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# MoE Transformer training

""" END OF THIS PART """

---

## Section B: Score-Based Generative Model (33 points)

### Background

Score-based generative models learn the **score function** $\nabla_x \log p(x)$ — the gradient of the log-probability with respect to the data. The score function points toward regions of higher data density.

**Noise Conditional Score Network (NCSN):**

Perturb data with Gaussian noise at multiple scales $\sigma_1 > \sigma_2 > \ldots > \sigma_K$:
$$\tilde{x} = x + \sigma_i \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

Train a network $s_\theta(x, \sigma)$ to estimate the score of the noisy distribution:
$$\mathcal{L}(\theta) = \sum_{i=1}^{K} \sigma_i^2 \, \mathbb{E}_{x \sim p_{\text{data}}, \epsilon \sim \mathcal{N}} \left[ \left\| s_\theta(\tilde{x}, \sigma_i) + \frac{\epsilon}{\sigma_i} \right\|^2 \right]$$

Note: the true score of the noisy distribution is $\nabla_{\tilde{x}} \log p_{\sigma_i}(\tilde{x}) = -\epsilon / \sigma_i$.

**Sampling via Langevin dynamics:**
$$x_{t+1} = x_t + \frac{\epsilon_t}{2} s_\theta(x_t, \sigma_i) + \sqrt{\epsilon_t} \, z_t, \quad z_t \sim \mathcal{N}(0, I)$$

Run Langevin dynamics for each noise level $\sigma_i$ from largest to smallest.

### B.1: Score Function Properties (5 points)

**[Non-coding]**

1. (2 points) For a 1D Gaussian $p(x) = \frac{1}{\sqrt{2\pi}\sigma} e^{-x^2/(2\sigma^2)}$, compute the score function $\nabla_x \log p(x)$.

2. (3 points) For a mixture of two Gaussians $p(x) = 0.5 \mathcal{N}(-2, 0.5^2) + 0.5 \mathcal{N}(2, 0.5^2)$, sketch the score function. Where does it point toward? Where is it zero?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

### B.2: Score Network (6 points)

**[Coding]** Implement the noise-conditional score network $s_\theta(x, \sigma)$.

- Input: concatenation of $x \in \mathbb{R}^d$ and $\sigma \in \mathbb{R}$
- Architecture: 3 hidden layers of 128 units with SiLU
- Output: $s \in \mathbb{R}^d$ (same dimension as $x$)
- **Important:** Divide the network output by $\sigma$ to help with multi-scale learning: $s_\theta(x, \sigma) = \frac{\text{network}(x, \sigma)}{\sigma}$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class ScoreNetwork(nn.Module):
    def __init__(self, d, hidden_dim=128, num_layers=3):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x, sigma):
        """
        Args:
            x: (B, d)
            sigma: (B, 1) or scalar
        Returns:
            score: (B, d)
        """
        pass  # YOUR CODE

""" END OF THIS PART """

### B.3: Denoising Score Matching Loss (8 points)

**[Coding]** Implement the training loss.

For a batch of data $x$:
1. Sample noise level $\sigma_i$ uniformly from the $K$ levels
2. Sample noise $\epsilon \sim \mathcal{N}(0, I)$
3. Create noisy data: $\tilde{x} = x + \sigma_i \epsilon$
4. Predict score: $\hat{s} = s_\theta(\tilde{x}, \sigma_i)$
5. Target: $s^* = -\epsilon / \sigma_i$
6. Loss: $\sigma_i^2 \|\hat{s} - s^*\|^2$ (averaged over batch)

Use $K = 10$ noise levels geometrically spaced from $\sigma_1 = 10$ to $\sigma_K = 0.01$.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def score_matching_loss(model, x, sigmas):
    """
    Args:
        model: ScoreNetwork
        x: (B, d) clean data
        sigmas: (K,) noise levels
    Returns:
        loss: scalar
    """
    pass  # YOUR CODE

""" END OF THIS PART """

### B.4: Annealed Langevin Dynamics Sampling (8 points)

**[Coding]** Implement the sampling algorithm.

For each noise level $\sigma_i$ from $\sigma_1$ (largest) to $\sigma_K$ (smallest):
- Run $T$ steps of Langevin dynamics with step size $\epsilon_i = \alpha \cdot \sigma_i^2 / \sigma_K^2$ where $\alpha$ is a small constant (e.g., $2 \times 10^{-5}$)
- Each step: $x \leftarrow x + \frac{\epsilon_i}{2} s_\theta(x, \sigma_i) + \sqrt{\epsilon_i} z$ where $z \sim \mathcal{N}(0, I)$

Use $T = 100$ steps per noise level.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

@torch.no_grad()
def annealed_langevin_sampling(model, sigmas, num_samples, d,
                                steps_per_level=100, alpha=2e-5):
    pass  # YOUR CODE

""" END OF THIS PART """

### B.5: Train and Evaluate (6 points)

**[Coding]** Train on a 2D mixture of 4 Gaussians (centers at $(\pm 3, \pm 3)$, $\sigma = 0.5$, 250 per cluster).

- 300 epochs, Adam lr=1e-3, batch_size=128
- Generate 500 samples after training
- Plot training data and generated samples side by side

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Train score model and generate samples

""" END OF THIS PART """

---

## Section C: Physics-Informed Learning for an Unknown PDE (33 points)

### Background

You are given a PDE that you may not have seen before: the **viscous Burgers' equation**:

$$u_t + u \cdot u_x = \nu u_{xx}$$

This is a nonlinear PDE (note the $u \cdot u_x$ term). It models fluid flow and can develop sharp gradients (shocks) for small $\nu$.

**Domain:** $t \in [0, 1]$, $x \in [-1, 1]$

**IC:** $u(0, x) = -\sin(\pi x)$

**BC:** $u(t, -1) = 0$, $u(t, 1) = 0$

**Viscosity:** $\nu = 0.01 / \pi$

This problem does NOT have a simple analytical solution. You will train a PINN and evaluate it numerically.

### C.1: PDE Residual (6 points)

**[Coding]** Implement the Burgers' equation residual: $u_t + u \cdot u_x - \nu u_{xx}$.

The nonlinear term $u \cdot u_x$ means you need the model output $u$ AND its derivative $u_x$, then multiply them.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def burgers_residual(model, tx, nu=0.01/math.pi):
    """
    Compute Burgers' equation residual: u_t + u * u_x - nu * u_xx
    
    Args:
        model: nn.Module mapping (B, 2) -> (B, 1)
        tx: (B, 2) with t in column 0, x in column 1
        nu: viscosity
    
    Returns:
        residual: (B, 1)
    """
    pass  # YOUR CODE

""" END OF THIS PART """

### C.2: Build and Train the PINN (10 points)

**[Coding]** Build a PINN for the Burgers' equation and train it.

- Network: 6 hidden layers of 128 units with Tanh (deeper/wider because the solution is more complex)
- 20,000 PDE collocation points, 200 IC points, 200 BC points
- Adam optimizer, lr=1e-3
- Train for 5000 epochs
- **Note:** The domain is $x \in [-1, 1]$, not $[0, 1]$
- Print loss every 1000 epochs

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Build and train Burgers' PINN

""" END OF THIS PART """

### C.3: Visualization (5 points)

**[Coding]** Visualize the learned solution.

1. (3 points) Create a heatmap of $u(t, x)$ on a 100×100 grid.
2. (2 points) Plot $u$ vs. $x$ at times $t = 0, 0.25, 0.5, 0.75, 1.0$ (5 curves on one plot). Does the solution develop a sharp gradient (shock)?

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Visualization

""" END OF THIS PART """

### C.4: Verify Boundary and Initial Conditions (4 points)

**[Coding]** Check how well the trained PINN satisfies the IC and BC.

1. (2 points) Compute the max IC error: $\max_x |u_\theta(0, x) - (-\sin(\pi x))|$
2. (2 points) Compute the max BC error: $\max_t |u_\theta(t, -1)|$ and $\max_t |u_\theta(t, 1)|$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Verify IC and BC

""" END OF THIS PART """

### C.5: L-BFGS Refinement (4 points)

**[Coding]** Apply L-BFGS for 50 iterations after the Adam training to refine the solution. Compare the PDE residual before and after L-BFGS.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# L-BFGS refinement

""" END OF THIS PART """

### C.6: Analysis (4 points)

**[Non-coding]**

1. (2 points) The Burgers' equation is nonlinear. How does the nonlinearity $u \cdot u_x$ affect the PINN training compared to the linear heat equation? What challenges does it introduce?

2. (2 points) For very small $\nu$ (e.g., $\nu = 10^{-4}$), the solution develops a near-discontinuity. Why would a standard PINN struggle with this, and what modifications might help?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

---

## End of Exam

**Stop your timer.** Record your total time.

### Self-Assessment

After completing (or running out of time), answer these questions:

1. How many parts did you complete (out of 14)?
2. Which section was hardest? Why?
3. Where did you spend the most time? Was it worthwhile?
4. Did you use stated results to continue when stuck?
5. What would you study more before the actual exam?

### Self-Assessment Answers:

